# 自研 Decision Model：监督微调、校准、评估与导出

## Goal
使用 Hugging Face encoder 和本项目的选项评分 head，完成 `choice`、`score`、`noul` 的微调与推理。每个 state/问题/选项配对独立编码，选项概率由 softmax 得到。输入编码、backbone、决策 head、objective 和解码分别实现，接口见 [architecture.md](../../docs/architecture.md)。Laya 仅作为思想参考，运行不依赖其 SDK。

默认用随机 tiny BERT 和 30 条合成样例验证流程。示例指标不代表领域效果；正式实验改用 `configs/decision/hf.yaml` 或自己的配置、领域数据和分组策略。基座 encoder 通过 [Hugging Face AutoModel](https://huggingface.co/docs/transformers/model_doc/auto) 加载，决策 head 从随机参数开始训练。


## Setup
本地先运行 `python -m pip install -e '.[decision,notebooks]'`。

Colab 将仓库放在 `/content/advai-notebooks`，设置下面的 `ROOT`，再将 `INSTALL_DEPENDENCIES=True`。正式训练的 `output_dir` 应放在持久化存储，避免运行时断开丢失 checkpoint。

Notebook 调用 `src/advai_notebooks/decision` 中的实现，与命令行共用流程。

In [ ]:
from pathlib import Path
import json
import sys
import tempfile
import subprocess

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), Path("/content/advai-notebooks"))
assert (ROOT / "pyproject.toml").is_file(), "请设置 ROOT 为本仓库路径"
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{ROOT}[decision,notebooks]"], check=True)

CONFIG_PATH = ROOT / "configs/decision/smoke.yaml"
# 正式实验：CONFIG_PATH = ROOT / "configs/decision/your_domain.yaml"
print("Repository:", ROOT)
print("Config:", CONFIG_PATH.relative_to(ROOT))

## Steps
### 1. 校验数据及任务
开启校准时需要 `train / validation / calibration / test` 四个独立 split；关闭校准后只需三个 split。同一 `group_id` 或完全相同的 state 不允许跨 split。任务与标签格式见同目录 README。


In [ ]:
from advai_notebooks.decision.config import load_config
from advai_notebooks.decision.data import load_data, summary

config = load_config(CONFIG_PATH, root=ROOT)
records, tasks = load_data(config.path(config.data.path), config.path(config.data.tasks))
data_summary = summary(records, tasks)
print(json.dumps(data_summary, ensure_ascii=False, indent=2))
print("One training record:")
print(json.dumps(next(r for r in records if r["split"] == "train"), ensure_ascii=False, indent=2))

### 2. 执行完整实验
计算基座测试结果、训练、按配置的验证指标选模，可选地在独立 calibration split 拟合温度，再测试并导出。基座 epoch 0 默认参与选模。

Smoke 每次建立独立输出目录。正式配置使用 YAML 的 `output_dir`；恢复时设置 `RESUME` 为该目录的 `latest`，保留原配置及计划 epoch 数。

默认 `loss: supervised` 对全部类型使用 CE，对 score 增加有序累计概率误差。`ce_smoke.yaml` 是纯 CE 对照，`rlcd_smoke.yaml` 是可选奖励优化实验；使用真实数据、相同训练预算和多个 seed 才能比较效果。


In [ ]:
import yaml
from advai_notebooks.decision.pipeline import run_experiment

RESUME = None
run_config = CONFIG_PATH
if config.model.id == "tiny":
    assert RESUME is None, "恢复 smoke 时请使用已保存的 generated config"
    parent = ROOT / "outputs/decision"
    parent.mkdir(parents=True, exist_ok=True)
    experiment_dir = Path(tempfile.mkdtemp(prefix="notebook-smoke-", dir=parent))
    parameters = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
    parameters["output_dir"] = str(experiment_dir / "run")
    run_config = experiment_dir / "config.yaml"
    run_config.write_text(yaml.safe_dump(parameters, sort_keys=False), encoding="utf-8")

report = run_experiment(run_config, root=ROOT, resume=RESUME)
export_dir = Path(report["export_dir"])
print("Export:", export_dir)
print("Best epoch:", report["best_epoch"])
if report["calibration"]:
    print("Calibration decisions by type:", report["calibration"]["samples_by_type"])
    print("Types using neutral fallback:", report["calibration"]["unfitted_types"])
else:
    print("Calibration disabled; identity temperatures exported.")

### 3. 查看测试集对比
以下结果均来自同一测试集；权重选择和温度拟合使用其他 split。Accuracy 对概率标签采用最高概率的 gold 类别，NLL/Brier 使用完整 gold 分布。ECE 衡量最高概率与实际正确率的差异。

合成样例和随机模型不能用于判断真实业务提升。校准可能改善也可能降低测试指标，应观察真实独立测试集上的表现。

In [ ]:
from IPython.display import Markdown, display

metric_names = ["accuracy", "nll", "brier", "ece", "normalized_score_mae"]
lines = ["| Model | " + " | ".join(metric_names) + " |", "|---|" + "---|" * len(metric_names)]
for key, label in [("baseline", "Baseline"), ("fine_tuned", "Selected"), ("fine_tuned_calibrated", "Calibrated")]:
    if report[key] is None:
        continue
    values = report[key]["overall"]
    lines.append("| " + label + " | " + " | ".join(f"{values[name]:.4f}" for name in metric_names) + " |")
display(Markdown("\n".join(lines)))

### 4. 检查训练历史
训练 objective 与验证 NLL 分别作图；基座 epoch 0 没有训练 loss。默认监督目标使用 CE + score 有序损失；纯 CE 和 RLCD 可独立实验，RLCD 使用 Brier/有序累计概率奖励。短 smoke 的曲线只证明日志与评估输出能生成。


In [ ]:
import matplotlib.pyplot as plt

history = json.loads((export_dir.parent / "history.json").read_text())
epochs = [entry["epoch"] for entry in history]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), constrained_layout=True)
for ax, values, title in [
    (axes[0], [entry["train_loss"] for entry in history], "Training objective"),
    (axes[1], [entry["validation"]["overall"]["nll"] for entry in history], "Validation NLL"),
]:
    points = [(epoch, value) for epoch, value in zip(epochs, values) if value is not None]
    ax.plot([p[0] for p in points], [p[1] for p in points], marker="o", color="#2563eb")
    ax.set(title=title, xlabel="Epoch", ylabel="Loss per decision", xticks=epochs)
    ax.grid(alpha=0.2)
fig.suptitle("Synthetic smoke run" if report["random_model"] else "Decision fine-tuning")
plt.show()

## Checks
### 5. 重载原生导出模型并执行三类决策
产物包含 `decision_config.json`、完整模型权重、encoder 配置、tokenizer、任务和校准参数。使用本项目加载器即可推理。`confidence` 是最高选项概率（保留 `answer_confidence` 兼容字段），score 是零基 level 的概率加权期望。

`system_one` 是统一推理入口：传入 state 和 task 时自动读取模型携带的任务定义；传入 `DecisionRequest` 列表时直接推理，不需要 gold。三类答案均包含 probabilities 和 confidence，noul 表示成立概率。

与 Embedding/Ranker 一致，初始化、加载、保存、输入处理和公开推理完整实现在 `model.py`，不再需要 provider 或 Bundle 层。旧 predict 方法与函数式推理入口保留兼容。


In [ ]:
from advai_notebooks.decision.data import requests_from_tasks
from advai_notebooks.decision.model import DecisionModel

model = DecisionModel.load(export_dir, device="cpu")
exported_tasks = model.task_definitions
task_id = next(iter(exported_tasks))
state = next(r["state"] for r in records if r["split"] == "test" and r["task"] == task_id)
result = model.system_one(state, task=task_id)
requests = requests_from_tasks(state, exported_tasks, task_id)
assert all("target" not in item for item in model.features(requests))
direct_answers = model.system_one(requests)
assert direct_answers == list(result["answers"].values())
reloaded = DecisionModel.load(export_dir, device="cpu")
assert reloaded.system_one(state, task=task_id) == result
for qid, answer in result["answers"].items():
    assert abs(sum(answer["probabilities"].values()) - 1.0) < 1e-6
    assert answer["confidence"] == answer["answer_confidence"] == max(answer["probabilities"].values())
assert {a["type"] for a in result["answers"].values()} == {"choice", "score", "noul"}
print(json.dumps(result, ensure_ascii=False, indent=2))
print("Model facade, label-free request and native export reload checks passed.")


## Next Steps
换成真实领域数据，按来源、意图或时间分组，覆盖任务类型与类别。配置普通 HF encoder 的 ID、revision、token 预算、pooling、head 宽度、冻结策略和设备。推理成本随选项数量增长。

命令行和续训示例见 [README](README.md)。CPU smoke 仅验证流程；真实 checkpoint 的领域效果、CUDA/MPS 精度和显存仍需在目标环境验证。

当前为独立选项评分 baseline。跨选项交互 head 尚未实现；新增组件需要同时扩展工厂、配置、导出与测试。


## 可选：发布到 Hugging Face

将 `REPO_ID` 改为自己的 `namespace/model`。默认 `UPLOAD_TO_HUB = False`，只离线验证发布包并保存清单；设置为 `True` 后才实际上传。实际上传前通过 `hf auth login` 或 `HF_TOKEN` 配置认证。仓库默认私有，许可证由使用者明确指定。详细说明见 [发布指南](../../docs/publishing.md)。

In [ ]:
from advai_notebooks.decision.publish import publish
from advai_notebooks.common.artifacts import write_json

REPO_ID = "your-org/your-decision"
UPLOAD_TO_HUB = False
PUBLIC = False
LICENSE = None
publication = publish(
    export_dir, REPO_ID,
    private=not PUBLIC, license=LICENSE, dry_run=not UPLOAD_TO_HUB,
)
publication_path = export_dir.parent / "publication.json"
write_json(publication_path, publication)
print("Publication:", publication["status"])
print("Files:", [row["path"] for row in publication["files"]])
print("Result / preview:", publication_path)
if publication.get("commit"):
    print("Pin this revision:", publication["commit"])